# Piloto 2026 · calibración y transferencia del bloque CO₂

## tl;dr

Este notebook reproduce el análisis `solubility_o2_nitrogen_boost_continuous_release` a escala piloto. Calibra por separado los lotes 2 y 3, reserva un ensayo A de cada lote como holdout, transfiere parámetros entre lotes y evalúa transferencia laboratorio↔piloto.

La interpretación del inicio sigue siendo condicional: no existe CO₂ preinoculación observado, no hay CO₂ disuelto manual y pH/O₂ sólo aparecen de forma discreta en el lote 3 desde 16 h. Los ceros artificiales anteriores al inicio de adquisición se excluyen.

## Contexto y métodos

### Supuestos clave

- Lote 1 (`26134–26136`) permanece fuera de la calibración por perfiles CO₂ no confiables.
- El tiempo cero es la primera muestra química y el final es `active_end`.
- MassView se convierte de Ln/min a g CO₂ L⁻¹ h⁻¹ usando 230 L y 22,414 L mol⁻¹.
- La normalización primaria fija el background y las ganancias por TK de la calibración piloto previa; se incluye sensibilidad sin normalizar.
- Los pulsos se ubican por interpolación del cruce de densidad 1040 kg m⁻³ y se marcan en todos los gráficos CO₂.
- Los parámetros upstream de fermentación quedan fijos en su calibración piloto; sólo se ajusta la capa CO₂.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display, Image

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "fermentation_model").exists():
    ROOT = ROOT.parent
if not (ROOT / "fermentation_model").exists():
    raise RuntimeError("Execute from the repository or a descendant directory")
sys.path.insert(0, str(ROOT / "fermentation_model"))
sys.path.insert(0, str(ROOT / "fermentation_model" / "pilot_2026"))

from pilot_2026 import run_co2_solubility_cross_lot_validation_2026 as analysis
result = analysis.run_analysis()
print("Resultados:", analysis.RESULTS_DIR.relative_to(ROOT))

## Datos

In [ ]:
display(result["inventory"].round(4))
display(result["sensor_audit"].round(4))
display(result["pulse_schedule"][[
    "matrix", "batch", "pulse_time_h", "schedule_proxy_time_h",
    "bracket_start_h", "bracket_end_h", "timing_source", "lot3_event_reconstructed"
]].round(3))

In [ ]:
display(Image(filename=analysis.FIGURE_DIR / "co2_data_overview.png"))
display(Image(filename=analysis.FIGURE_DIR / "sensor_normalization_and_filter.png"))

In [ ]:
display(Image(filename=analysis.FIGURE_DIR / "temperature_profiles_model_input.png"))
display(Image(filename=analysis.FIGURE_DIR / "chemistry_and_pulse_timeline.png"))

## Resultados de calibración y validación

In [ ]:
display(result["fit_parameters"].round(5))
display(result["validation"][[
    "calibration_matrix", "target_matrix", "batch", "role", "rmse_g_l_h",
    "r2", "observed_onset_h", "predicted_onset_h", "onset_delay_h",
    "integral_ratio_pred_over_obs"
]].round(4))

In [ ]:
display(Image(filename=analysis.FIGURE_DIR / "calibration_overlays_pilot_lot2.png"))
display(Image(filename=analysis.FIGURE_DIR / "calibration_overlays_pilot_lot3.png"))
display(Image(filename=analysis.FIGURE_DIR / "heldout_cross_lot_validation.png"))

In [ ]:
display(result["model_validation"][[
    "model", "calibration_matrix", "target_matrix", "batch", "role", "rmse_g_l_h", "onset_delay_h"
]].round(4))
display(Image(filename=analysis.FIGURE_DIR / "heldout_model_comparison.png"))
display(Image(filename=analysis.FIGURE_DIR / "parameter_comparison.png"))

## Identificabilidad práctica

In [ ]:
display(result["jacobian"].round(5))
display(result["loo_summary"].round(5))
activation_corr = result["local_correlations"].query(
    "parameter_1 == 'chem_activation_start_fraction' and parameter_2 == 'chem_activation_duration_fraction'"
)
display(activation_corr.round(5))
display(Image(filename=analysis.FIGURE_DIR / "activation_identifiability_profiles.png"))

## Transferencia entre escalas y cobertura auxiliar

In [ ]:
display(result["cross_scale_metrics"][[
    "transfer_direction", "calibration_matrix", "target_matrix", "batch", "role", "rmse_g_l_h", "onset_delay_h"
]].round(4))
display(Image(filename=analysis.FIGURE_DIR / "cross_scale_transfer.png"))
display(Image(filename=analysis.FIGURE_DIR / "ph_do_availability.png"))

## Takeaways

La validación cruzada entre lotes prueba si el bloque de liberación/observación CO₂ es estable frente a cambios de lote y programa térmico. La transferencia laboratorio↔piloto evalúa el mismo bloque condicionalmente a los drivers upstream específicos de cada escala.

Los diagnósticos de Jacobiano, perfiles y leave-one-batch-out deben interpretarse con cautela: cada calibración por lote utiliza sólo dos fermentaciones después de reservar el holdout. Esta reproducción no transforma los ceros artificiales en evidencia de ausencia de fermentación y no usa el pH/O₂ tardío para localizar el inicio.

In [ ]:
print("Notebook ejecutado sin errores.")
print("Figuras:", len(list(analysis.FIGURE_DIR.glob("*.png"))))
print("Cross-scale:", result["manifest"]["cross_scale_status"])
print("Artefactos:", analysis.RESULTS_DIR.relative_to(ROOT))